## Chain rule

The chain rule tells you how to differentiate a **composition** of functions.

If `y = f(g(x))`, set `u = g(x)`. Then:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

In words: **multiply the derivatives along the chain.** The rate at which `y` changes
with `x` is the rate `y` changes with `u`, times the rate `u` changes with `x`.

For a longer chain `y = f(g(h(x)))` you just keep multiplying:

$$\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dv}\cdot\frac{dv}{dx}$$

**Why it matters for us:** backprop *is* the chain rule. A neural net is one big
composition of functions; to get the gradient of the loss w.r.t. any weight, you
multiply local derivatives backward from the loss to that weight.

In [5]:
import numpy as np

# Example: y = sin(x**2).  Let u = x**2, so y = sin(u).
#   dy/du = cos(u) = cos(x**2)
#   du/dx = 2x
#   dy/dx = cos(x**2) * 2x     <-- chain rule

def y(x):
    return np.sin(x**2)

def dy_dx_analytic(x):
    return np.cos(x**2) * 2*x

# Numerical check via central finite difference: (f(x+h) - f(x-h)) / 2h
def dy_dx_numeric(x, h=1e-5):
    return (y(x + h) - y(x - h)) / (2*h)

x = 1.3
print('analytic:', dy_dx_analytic(x))
print('numeric :', dy_dx_numeric(x))

analytic: -0.3091960801711924
numeric : -0.3091960803947025


## Partial derivatives

For a function of **several variables**, there's no single slope — it depends on which
direction you move. So we take the derivative **one variable at a time**, holding the
others fixed. The curly `∂` (vs. `d`) is the reminder that "everything else is frozen":

$$\frac{\partial f}{\partial x} = \text{rate } f \text{ changes as } x \text{ moves, with all other inputs held constant}$$

**Example:** `f(x, z) = x**2 * z + sin(z)`

- `∂f/∂x`: treat `z` as constant → `2xz`  (the `sin(z)` term is constant in `x`, so → 0)
- `∂f/∂z`: treat `x` as constant → `x**2 + cos(z)`

**The gradient** stacks all partials into a vector, pointing in the direction of
steepest increase (gradient descent steps along its negative):

$$\nabla f = \left[\frac{\partial f}{\partial x},\ \frac{\partial f}{\partial z}\right]$$

**Why it matters for us:** the loss depends on millions of weights. The gradient is the
vector of partials w.r.t. every weight — each computed by the chain rule while holding
the others fixed. That whole vector is what backprop produces.

In [6]:
# f(x, z) = x**2 * z + sin(z)
def f(x, z):
    return x**2 * z + np.sin(z)

# Analytic partials
def grad_analytic(x, z):
    df_dx = 2*x*z
    df_dz = x**2 + np.cos(z)
    return np.array([df_dx, df_dz])

# Numeric gradient: central difference, perturbing ONE variable at a time.
def grad_numeric(x, z, h=1e-5):
    df_dx = (f(x + h, z) - f(x - h, z)) / (2*h)   # z held fixed
    df_dz = (f(x, z + h) - f(x, z - h)) / (2*h)   # x held fixed
    return np.array([df_dx, df_dz])

x, z = 1.3, 0.7
print('analytic grad:', grad_analytic(x, z))
print('numeric  grad:', grad_numeric(x, z))

analytic grad: [1.82       2.45484219]
numeric  grad: [1.82       2.45484219]


## Gradients of scalar functions

A **scalar function** maps a vector to a single number: `f: ℝⁿ → ℝ`. Think of the loss:
many inputs (all the weights), one output (the loss value).

Its **gradient** is the vector holding the partial derivative w.r.t. *each* input:

$$\nabla f(\mathbf{x}) = \left[\frac{\partial f}{\partial x_1},\ \frac{\partial f}{\partial x_2},\ \dots,\ \frac{\partial f}{\partial x_n}\right]$$

Key facts:

- **Same shape as the input.** If `x` is a length-`n` vector, `∇f` is also length `n`.
  (If `x` is a matrix of weights, `∇f` is a matrix of the same shape.)
- **Direction of steepest ascent.** `∇f` points where `f` increases fastest; `−∇f` is the
  descent direction. Gradient descent is literally `x ← x − lr · ∇f`.
- **Zero at flat points.** At a minimum/maximum/saddle, `∇f = 0`.

**The numeric gradient checker (generalized).** The per-variable loop from before becomes:
perturb each component `x_i` by `±h`, hold the rest fixed, take a central difference. This
gives a slow but formula-free gradient — the tool we'll use to verify every backward pass
in this project.

In [7]:
# A scalar function of a VECTOR:  f(x) = sum_i (x_i**2)  +  x_0 * x_1
#   -> maps R^n to a single number.
def f(x):
    return np.sum(x**2) + x[0] * x[1]

# Analytic gradient (worked out by hand, component by component):
#   d/dx_i of sum(x^2) = 2 x_i
#   the extra x0*x1 term adds x1 to component 0, and x0 to component 1
def grad_analytic(x):
    g = 2 * x.copy()
    g[0] += x[1]
    g[1] += x[0]
    return g

# General numeric gradient checker: loops over EVERY component,
# perturbs just that one by +/- h, central difference. Works for any f: R^n -> R.
def numeric_gradient(f, x, h=1e-5):
    grad = np.zeros_like(x, dtype=float)
    for i in range(x.size):
        step = np.zeros_like(x, dtype=float)
        step[i] = h
        grad[i] = (f(x + step) - f(x - step)) / (2*h)
    return grad

x = np.array([1.0, 2.0, -3.0, 0.5])
print('analytic:', grad_analytic(x))
print('numeric :', numeric_gradient(f, x))
print('max abs diff:', np.max(np.abs(grad_analytic(x) - numeric_gradient(f, x))))

analytic: [ 4.  5. -6.  1.]
numeric : [ 4.  5. -6.  1.]
max abs diff: 1.2812417793384157e-10


## Softmax

**Softmax** turns a vector of arbitrary real numbers (**logits**) into a **probability
distribution** — all entries in `(0, 1)` and summing to `1`:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

How to read it:

- **Exponentiate** every logit → all values become positive.
- **Normalize** by the sum → they become probabilities that add to 1.
- It's **monotonic**: the largest logit gets the largest probability. It's a *soft* argmax —
  instead of picking one winner, it spreads weight, mostly onto the top entries.

Properties that matter later:

- **Shift-invariant:** adding a constant to every logit doesn't change the output (this is
  exactly what the log-sum-exp trick exploits below).
- **Not scale-invariant:** multiplying logits by a factor sharpens (large factor → near
  one-hot) or flattens (small factor → near uniform) the distribution. That factor is the
  **temperature** knob used in sampling.

**Why it matters for us:** it's the output layer of a classifier / language model — it
converts the network's raw scores into `p(class)` or `p(next token)`, and it pairs with
cross-entropy loss (Step 2).

In [9]:
# Softmax: logits -> probability distribution
def softmax(x):
    e = np.exp(x - np.max(x))   # stable form (see next section for why)
    return e / np.sum(e)

logits = np.array([2.0, 1.0, 0.1])
p = softmax(logits)
print('probs      :', p)
print('sum to 1   :', p.sum())
print('argmax kept:', np.argmax(logits) == np.argmax(p))   # largest logit -> largest prob

# Temperature: scaling logits sharpens or flattens the distribution
for T in [0.5, 1.0, 5.0]:
    print(f'T={T}: {np.round(softmax(logits / T), 3)}')   # small T -> peaky, large T -> flat

probs      : [0.65900114 0.24243297 0.09856589]
sum to 1   : 1.0
argmax kept: True
T=0.5: [0.864 0.117 0.019]
T=1.0: [0.659 0.242 0.099]
T=5.0: [0.4   0.327 0.273]


## Numerical stability: floating point, overflow, log-sum-exp

### Floating point
Computers store reals in finite bits (float64 = 64 bits). Two consequences:

- **Finite range.** The largest float64 is about `1.8e308`. Anything bigger becomes `inf`
  (**overflow**); anything tiny underflows to `0.0`.
- **Finite precision.** Only ~15–16 significant decimal digits. `0.1 + 0.2 != 0.3` exactly,
  and subtracting two nearly-equal numbers throws away digits (**catastrophic cancellation**).

### Where it bites us: softmax
Softmax needs exponentials: `softmax(x)_i = exp(x_i) / Σ_j exp(x_j)`. But `exp` grows
insanely fast — `exp(1000)` is `inf` in float64. Then `inf / inf = nan` and the whole
forward pass is poisoned. Logits in a real net easily reach these ranges.

### The log-sum-exp trick
Softmax is **shift-invariant**: subtracting the same constant `c` from every logit leaves
the result unchanged, because the constant cancels top and bottom:

$$\frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{-c}\,e^{x_i}}{e^{-c}\sum_j e^{x_j}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Choose `c = max(x)`. Now the largest exponent is `exp(0) = 1` — no overflow — and every
other term is between 0 and 1. Same math, safe numbers.

The same idea gives a stable **log-sum-exp**, `log Σ exp(x_j) = c + log Σ exp(x_j − c)`,
which is how cross-entropy loss is computed directly from logits (never forming the raw
`exp`).

In [10]:
# Floating-point limits
print('largest float64 :', np.finfo(np.float64).max)
print('exp(1000)       :', np.exp(1000.0))      # -> inf (overflow)
print('0.1 + 0.2 == 0.3:', 0.1 + 0.2 == 0.3)    # -> False (finite precision)

# Naive softmax: overflows on large logits
def softmax_naive(x):
    e = np.exp(x)
    return e / np.sum(e)

# Stable softmax: subtract the max first (log-sum-exp trick)
def softmax_stable(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

x = np.array([1000.0, 1001.0, 1002.0])   # big logits
print('\nnaive :', softmax_naive(x))     # -> [nan nan nan]
print('stable:', softmax_stable(x))      # -> correct, finite

# Same answer as naive on SMALL inputs (shift-invariance, sanity check)
small = np.array([1.0, 2.0, 3.0])
print('\nagree on small inputs:', np.allclose(softmax_naive(small), softmax_stable(small)))

# Stable log-sum-exp: log(sum(exp(x))) without ever forming exp(x) directly
def logsumexp(x):
    c = np.max(x)
    return c + np.log(np.sum(np.exp(x - c)))

print('logsumexp(big logits):', logsumexp(x))   # finite, ~1002.4

largest float64 : 1.7976931348623157e+308
exp(1000)       : inf
0.1 + 0.2 == 0.3: False

naive : [nan nan nan]
stable: [0.09003057 0.24472847 0.66524096]

agree on small inputs: True
logsumexp(big logits): 1002.4076059644444


/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:3: RuntimeWarning: overflow encountered in exp
  This is separate from the ipykernel package so we can avoid doing imports until
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:8: RuntimeWarning: overflow encountered in exp
  
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:9: RuntimeWarning: invalid value encountered in true_divide
  if __name__ == "__main__":


## Backprop

Training = minimize a scalar loss `L` over millions of parameters via gradient descent,
which needs `∂L/∂θ` for every parameter `θ`. **Backprop** computes all of them exactly in
**one forward + one backward pass** (vs. `numeric_gradient`, which needs 2 passes *per
parameter*).

It's the **chain rule applied backward** through the network:

- **Forward:** run input → output, caching intermediate values.
- **Backward:** start at the loss (`∂L/∂L = 1`) and walk backward. Each layer receives the
  upstream gradient `∂L/∂(its output)` and returns `∂L/∂(params)` (to update) and
  `∂L/∂(its input)` (passed to the previous layer).

Efficient because the loss is a single scalar, so one backward sweep reaches every
parameter (reverse-mode autodiff). `numeric_gradient` stays as the checker that verifies
each hand-derived backward is correct.

## Activations: sigmoid, tanh, ReLU, GELU

Without a nonlinearity between linear layers, the whole stack collapses to one linear map.
Activations add the curvature that lets a net approximate non-linear functions.

| Name | Formula | Range | Derivative |
|---|---|---|---|
| **sigmoid** | `1 / (1 + e^-x)` | (0, 1) | `s(x)·(1 − s(x))` |
| **tanh** | `(e^x − e^-x)/(e^x + e^-x)` | (−1, 1) | `1 − tanh(x)²` |
| **ReLU** | `max(0, x)` | [0, ∞) | `1 if x>0 else 0` |
| **GELU** | `x · Φ(x)` (Φ = normal CDF) | ≈[−0.17, ∞) | smooth, ≈ReLU |

- **sigmoid / tanh** squash into a fixed range but **saturate** (flat tails → tiny
  gradients). tanh is zero-centered, usually preferred over sigmoid for hidden layers.
- **ReLU** is cheap and doesn't saturate for `x>0` → the default for deep nets.
- **GELU** is a smooth ReLU used in Transformers (what we'll use).

In [ ]:
import numpy as np

def sigmoid(x):   return 1 / (1 + np.exp(-x))
def d_sigmoid(x): s = sigmoid(x); return s * (1 - s)

def tanh(x):      return np.tanh(x)
def d_tanh(x):    return 1 - np.tanh(x) ** 2

def relu(x):      return np.maximum(0, x)
def d_relu(x):    return (x > 0).astype(float)

# GELU (tanh approximation) and its derivative
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

# Check a derivative numerically (central difference) at a point
x0 = 0.7
num = (sigmoid(x0 + 1e-5) - sigmoid(x0 - 1e-5)) / (2e-5)
print('d_sigmoid analytic:', d_sigmoid(x0))
print('d_sigmoid numeric :', num)

## Weight initialization (why scale matters: Xavier/He)

Weights start random (to break symmetry) but their **scale** is critical. Each layer
multiplies its input by `W`; if `W` is too large the signal (and gradients) **explode**
across layers, too small and they **vanish**.

The fix: scale the initial variance by the layer width so signal magnitude is preserved
layer to layer:

- **Xavier/Glorot** — `Var(W) = 1 / n_in` (or `2/(n_in+n_out)`). For tanh/sigmoid.
- **He** — `Var(W) = 2 / n_in`. For ReLU (accounts for ReLU zeroing half the inputs).

Practically: `W = randn(n_in, n_out) * sqrt(scale / n_in)`.

## Vanishing / exploding gradients

Backprop multiplies many local derivatives along the path from loss to an early layer. If
those factors are consistently **< 1**, the product shrinks toward 0 (**vanishing** — early
layers barely learn); if **> 1**, it blows up (**exploding** — unstable, `nan`).

This is why deep nets were historically hard to train. Mitigations we'll use:
- **ReLU/GELU** (derivative 1 for active units, doesn't saturate)
- **good init** (Xavier/He keeps factors near 1)
- **residual connections** and **LayerNorm** (Step 5) — keep gradients flowing through depth.

## Universal approximation theorem

A feedforward net with a **single hidden layer** and a nonlinear activation can approximate
*any* continuous function on a bounded region, to arbitrary accuracy — given enough hidden
units.

Caveats worth knowing:
- It's an **existence** result: it says such weights exist, not that gradient descent will
  find them.
- "Enough units" can be impractically large. **Depth** (many layers) expresses many
  functions far more efficiently than one very wide layer — which is why we go deep.

## Dead ReLUs & activation saturation

Two failure modes where neurons stop learning because their **gradient is ~0**:

- **Dead ReLU** — if a ReLU's input is always negative, its output is always 0 and its
  derivative is always 0, so no gradient flows and the weights never update — the neuron is
  permanently "dead." Caused by bad init or too-large learning rates. Mitigated by careful
  init, lower lr, or variants (LeakyReLU, GELU).
- **Saturation** — sigmoid/tanh have flat tails; for large `|x|` the derivative → 0, so
  gradients vanish there too. Another reason ReLU-family activations are preferred in the
  hidden layers of deep nets.